In [92]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [93]:
from testgen.prompts import Sensors

labels = Sensors.split("\n")
for i, sensor in enumerate(labels):
    labels[i] = sensor[sensor.find("(")+1: sensor.find(")")].strip()
    
labels = np.array(labels)

labels

array(['Acc', 'WSA', 'WS', 'YR', 'ST'], dtype='<U3')

In [94]:
# read results file
base_path = Path().cwd()
results_path = base_path.parent / "results"

available_results = list(results_path.glob("bulk*.json"))

# drop file with lower accuracy if multiple are found
available_results_dict = {}

for i, f in enumerate(available_results):
    t_ = f.name.split("_acc-")
    if available_results_dict.get(t_[0]) is None:
        available_results_dict[t_[0]] = (i, float(t_[1].split("_")[0]))
    else:
        if available_results_dict[t_[0]][1] < float(t_[1].split("_")[0]):
            del available_results[available_results_dict[t_[0]][0]]
            available_results_dict[t_[0]] = (i, float(t_[1].split("_")[0]))
        

available_results

[PosixPath('/mnt/d/scripts/hil/hil-test-case-gen/results/bulk-3_gpt-4o-mini_n-1_acc-0.828_10.29.2024-09:36:37.json'),
 PosixPath('/mnt/d/scripts/hil/hil-test-case-gen/results/bulk-5_gpt-4o-mini_n-1_acc-0.854_10.29.2024-09:31:56.json'),
 PosixPath('/mnt/d/scripts/hil/hil-test-case-gen/results/bulk-8_gpt-4o-mini_n-1_acc-0.832_10.29.2024-09:38:32.json')]

In [95]:
def calc_scores(df):
    df_scores = pd.DataFrame(
        columns=["sensor", "accuracy", "precision", "recall", "f1"]
    )

    y_true = df["true_label"]
    y_pred = df["pred_label"]
    unique_labels = np.unique(y_true)
    unique_labels.sort()

    accuracy_score_ = round(accuracy_score(y_true, y_pred), 2)
    precision_score_ = round(
        precision_score(y_true, y_pred, average="weighted", labels=unique_labels), 2
    )
    recall_score_ = round(
        recall_score(y_true, y_pred, average="weighted", labels=unique_labels), 2
    )
    f1_score_ = round(
        f1_score(y_true, y_pred, average="weighted", labels=unique_labels), 2
    )

    df_scores.loc[df_scores.shape[0] + 1] = [
        "All",
        accuracy_score_,
        precision_score_,
        recall_score_,
        f1_score_,
    ]

    for label in unique_labels:
        t_ = df[df["true_label"] == label]

        y_true_ = t_["true_label"]
        y_pred_ = t_["pred_label"]

        accuracy_score_ = round(accuracy_score(y_true_, y_pred_), 2)
        precision_score_ = round(
            precision_score(y_true_, y_pred_, average="weighted", labels=[label]), 2
        )
        recall_score_ = round(
            recall_score(y_true_, y_pred_, average="weighted", labels=[label]), 2
        )
        f1_score_ = round(
            f1_score(y_true_, y_pred_, average="weighted", labels=[label]), 2
        )

        df_scores.loc[df_scores.shape[0] + 1] = [
            label,
            accuracy_score_,
            precision_score_,
            recall_score_,
            f1_score_,
        ]

    return df_scores.set_index("sensor")

In [96]:
def analyze(filename):

    type_, model_name, number_examples, *_ = filename.stem.split("_")
    number_examples = int(number_examples.split("-")[-1])
    
    type_, type_n = type_.split("-")
    type_n = int(type_n)

    file_under_investigation = results_path / filename
    with file_under_investigation.open("r") as f:
        data = json.load(f)
        
    # split responses and general stats
    responses = pd.DataFrame()
    for res in data["responses"]:
        responses = pd.concat([responses, pd.DataFrame(res)])
    responses.set_index("idx", inplace=True)
    del data["responses"]

    # collect stats per experiment
    stats = pd.DataFrame({k: [v]for k,v in data.items()})
    stats.insert(0, "type", type_)
    stats.insert(1, "type_n", type_n)
    stats.insert(2, "model_name", model_name)
    stats.insert(3, "number_examples", number_examples)

    # collect label names for predictions and ground truth
    responses["true_label"] = responses["true_vector"].map(lambda x: " & ".join(labels[np.array(x[1:-1].split(",")).astype(bool)]) )
    responses["pred_label"] = responses["pred_vector"].map(lambda x: " & ".join(labels[np.array(x[1:-1].split(",")).astype(bool)]) )

    # accuracy per sensor
    summarize_ = responses.groupby(["true_label"]).aggregate({
        "accuracy": "sum",
        "true_label": "count"
    })

    summarize_.columns = ["true", "total"]
    summarize_["false"] = summarize_["total"] - summarize_["true"]


    all_vals = summarize_.sum(axis=0).values.tolist()
    summarize_.loc["All"] = all_vals

    summarize_.insert(0, "type", type_)
    summarize_.insert(1, "type_n", type_n)
    summarize_.insert(2, "model_name", model_name)
    summarize_.insert(3, "number_examples", number_examples)

    df_scores = calc_scores(responses)

    summarize_ = summarize_.merge(df_scores, left_index=True, right_index=True).reset_index(names="sensors")
    
    return stats, summarize_

In [97]:
def plot_summarize(summarize_):
    ax = summarize_["accuracy"].plot.bar(
        title="Accuracy per Sensor",
        xlabel="",
    )

    _ = ax.bar_label(ax.containers[0])

In [98]:
stats = pd.DataFrame()
summarize = pd.DataFrame()

for filename in available_results:
    stats_, summarize_ = analyze(filename)
    
    stats = pd.concat([stats, stats_])
    summarize = pd.concat([summarize, summarize_])
    

stats.drop(columns="examples", inplace=True)
stats.set_index("number_examples", inplace=True)

summarize.reset_index(drop=True, inplace=True)

In [99]:
stats

,type,type_n,model_name,accuracy,number_of_reqs,total_tokens,total_completion_tokens,avg_token_per_req,avg_completion_token_per_req,avg_time_per_req
number_examples,,,,,,,,,,
1,bulk,3,gpt-4o-mini,0.827957,62,56890,2976,305.860215,16.0,0.354957
1,bulk,5,gpt-4o-mini,0.854054,37,36510,2960,197.351351,16.0,0.155577
1,bulk,8,gpt-4o-mini,0.831522,23,25877,2944,140.635870,16.0,0.162586


In [100]:
summarize

,sensors,type,type_n,model_name,number_examples,true,total,false,accuracy,precision,recall,f1
0,Acc,bulk,3,gpt-4o-mini,1,30,31,1,0.97,1.00,0.97,0.98
1,Acc & WSA,bulk,3,gpt-4o-mini,1,19,24,5,0.79,1.00,0.79,0.88
2,ST,bulk,3,gpt-4o-mini,1,39,45,6,0.87,1.00,0.87,0.93
3,WS,bulk,3,gpt-4o-mini,1,26,26,0,1.00,1.00,1.00,1.00
4,WSA,bulk,3,gpt-4o-mini,1,14,33,19,0.42,1.00,0.42,0.60
5,YR,bulk,3,gpt-4o-mini,1,26,27,1,0.96,1.00,0.96,0.98
6,All,bulk,3,gpt-4o-mini,1,154,186,32,0.83,0.92,0.83,0.86
7,Acc,bulk,5,gpt-4o-mini,1,27,30,3,0.90,1.00,0.90,0.95
8,Acc & WSA,bulk,5,gpt-4o-mini,1,19,25,6,0.76,1.00,0.76,0.86
9,ST,bulk,5,gpt-4o-mini,1,35,45,10,0.78,1.00,0.78,0.88
